# Day 07: DPO (Direct Preference Optimization) 实战

**学习目标：**
- 理解 DPO 的核心洞见：用一个损失函数替代 RLHF 的三阶段流程
- 掌握偏好数据的格式和质量标准
- 完成 DPO 实战训练（加载模型→准备数据→训练→评估）
- 对比 Base / SFT / DPO 三种模型的效果差异

---

## 1. DPO 核心洞见：从 RLHF 到 DPO

### 1.1 RLHF 的痛点

RLHF 虽然有效，但有三个显著痛点：

| 痛点 | 具体问题 |
|------|----------|
| **复杂度** | 需要训练 3 个模型（SFT、RM、PPO策略） |
| **不稳定** | PPO 对超参数极其敏感，训练经常崩溃 |
| **资源消耗** | PPO 训练需要同时加载策略模型、参考模型、奖励模型、critic模型 |

> **DPO 的核心问题：** 能否跳过奖励模型和 PPO，直接从偏好数据学习？

### 1.2 DPO 的数学直觉（不需要证明）

DPO 的推导基于一个关键观察：

**Step 1:** RLHF 的最优策略有封闭解：
$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{ref}(y|x) \cdot \exp\left(\frac{r(x,y)}{\beta}\right)$$

**Step 2:** 反过来，奖励可以从策略中恢复：
$$r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

**Step 3:** 代入 Bradley-Terry 模型，$Z(x)$ 被消掉，得到 DPO 损失！

**直觉：** 不需要先学一个奖励函数再优化策略，可以**直接用偏好数据优化策略**，因为最优策略和最优奖励函数之间存在一一对应关系。

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps,
             beta=0.1):
    """
    DPO 损失函数 — 核心只有 10 行！
    
    参数:
        policy_chosen_logps:   当前模型对 chosen 回复的对数概率
        policy_rejected_logps: 当前模型对 rejected 回复的对数概率
        ref_chosen_logps:      参考模型对 chosen 回复的对数概率
        ref_rejected_logps:    参考模型对 rejected 回复的对数概率
        beta: 温度参数，控制偏离参考模型的程度
    """
    # 计算对数概率比（隐式奖励）
    chosen_logratios = policy_chosen_logps - ref_chosen_logps      # 隐式奖励(chosen)
    rejected_logratios = policy_rejected_logps - ref_rejected_logps  # 隐式奖励(rejected)
    
    # DPO 损失: -log σ(β * (r_chosen - r_rejected))
    logits = beta * (chosen_logratios - rejected_logratios)
    loss = -F.logsigmoid(logits).mean()
    
    # 额外指标
    chosen_rewards = beta * chosen_logratios.detach()
    rejected_rewards = beta * rejected_logratios.detach()
    reward_margin = (chosen_rewards - rejected_rewards).mean().item()
    accuracy = (chosen_rewards > rejected_rewards).float().mean().item()
    
    return loss, reward_margin, accuracy

# 演示
batch_size = 8
torch.manual_seed(42)

# 模拟对数概率
policy_chosen = torch.randn(batch_size) - 1    # 当前模型对 chosen 的 logP
policy_rejected = torch.randn(batch_size) - 2  # 当前模型对 rejected 的 logP
ref_chosen = torch.randn(batch_size) - 1.5     # 参考模型对 chosen 的 logP
ref_rejected = torch.randn(batch_size) - 1.5   # 参考模型对 rejected 的 logP

loss, margin, acc = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)
print(f"DPO Loss:       {loss.item():.4f}")
print(f"Reward Margin:  {margin:.4f}  (chosen - rejected, 越大越好)")
print(f"Accuracy:       {acc:.2%}  (chosen reward > rejected reward 的比例)")

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# RLHF vs DPO 流程对比图
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# RLHF 流程
ax = axes[0]
ax.axis("off")
ax.set_title("RLHF 流程（复杂）", fontsize=13, fontweight="bold", color="#FF6B6B")
steps_rlhf = [
    (0.5, 0.8, "SFT 训练", "#4ECDC4"),
    (0.5, 0.6, "收集偏好数据", "#FFE66D"),
    (0.5, 0.4, "训练 Reward Model", "#FF6B6B"),
    (0.5, 0.2, "PPO 优化 (4个模型)", "#45B7D1"),
]
for i, (x, y, text, color) in enumerate(steps_rlhf):
    ax.add_patch(plt.Rectangle((0.1, y - 0.07), 0.8, 0.12, facecolor=color, alpha=0.7, transform=ax.transAxes))
    ax.text(x, y, f"Step {i+1}: {text}", ha="center", va="center", transform=ax.transAxes, fontsize=10)
    if i < len(steps_rlhf) - 1:
        ax.annotate("", xy=(0.5, steps_rlhf[i+1][1] + 0.07), xytext=(0.5, y - 0.07),
                    arrowprops=dict(arrowstyle="->", lw=1.5), xycoords="axes fraction", textcoords="axes fraction")

# DPO 流程
ax = axes[1]
ax.axis("off")
ax.set_title("DPO 流程（简洁）", fontsize=13, fontweight="bold", color="#4ECDC4")
steps_dpo = [
    (0.5, 0.7, "SFT 训练", "#4ECDC4"),
    (0.5, 0.4, "DPO 直接优化\n(只需偏好数据 + 1个损失函数)", "#4ECDC4"),
]
for i, (x, y, text, color) in enumerate(steps_dpo):
    h = 0.12 if i == 0 else 0.18
    ax.add_patch(plt.Rectangle((0.1, y - h/2), 0.8, h, facecolor=color, alpha=0.7, transform=ax.transAxes))
    ax.text(x, y, f"Step {i+1}: {text}", ha="center", va="center", transform=ax.transAxes, fontsize=10)
    if i < len(steps_dpo) - 1:
        ax.annotate("", xy=(0.5, steps_dpo[i+1][1] + 0.09), xytext=(0.5, y - 0.06),
                    arrowprops=dict(arrowstyle="->", lw=1.5), xycoords="axes fraction", textcoords="axes fraction")
ax.text(0.5, 0.15, "去掉了 Reward Model 和 PPO!", ha="center", va="center",
        transform=ax.transAxes, fontsize=12, color="red", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 2. 偏好数据详解

### 2.1 数据格式

DPO 的每条训练样本包含三个字段：

```json
{
    "prompt": "用户的问题/指令",
    "chosen": "人类偏好的回复（好的）",
    "rejected": "人类不偏好的回复（差的）"
}
```

**关键：** chosen 和 rejected 是**同一个 prompt** 的两个回复，形成一个偏好对。

In [ ]:
# 偏好数据示例
preference_examples = [
    {
        "prompt": "请解释什么是机器学习？",
        "chosen": "机器学习是人工智能的一个分支，它使计算机能够从数据中自动学习和改进。"
                  "主要分为监督学习、无监督学习和强化学习三大类。"
                  "监督学习使用标注数据训练；无监督学习发现隐藏模式；强化学习通过奖励信号学习。",
        "rejected": "机器学习就是让机器学东西。",
    },
    {
        "prompt": "Python 和 Java 有什么区别？",
        "chosen": "主要区别：1) 类型系统 — Python 动态类型，Java 静态类型；"
                  "2) 语法 — Python 用缩进，Java 用大括号；"
                  "3) 执行方式 — Python 解释执行，Java 编译为字节码在 JVM 上运行。",
        "rejected": "一个用缩进一个用大括号。",
    },
]

for i, ex in enumerate(preference_examples):
    print(f"=== 样本 {i+1} ===")
    print(f"Prompt:   {ex['prompt']}")
    print(f"Chosen:   {ex['chosen'][:80]}...")
    print(f"Rejected: {ex['rejected']}")
    print()

### 2.2 数据来源

| 来源 | 代表数据集 | 特点 |
|------|-----------|------|
| **人工标注** | Anthropic HH-RLHF | 质量最高，成本高 |
| **AI 辅助** | UltraFeedback | GPT-4 打分，成本低 |
| **模型对比** | Chatbot Arena | 真实用户 A/B 测试 |
| **自动生成** | Self-play | 模型生成 + 自动评估 |

### 2.3 数据质量标准

偏好数据的质量直接决定 DPO 的效果：

- **清晰的偏好差异** — chosen 和 rejected 之间的差距要明显
- **多样性** — 覆盖不同类型的 prompt 和回复风格
- **一致性** — 标注标准统一，避免矛盾标注
- **合理的难度** — 太简单的偏好（乱码 vs 正常）学不到有用信息

In [ ]:
# 数据质量检查示例
def check_preference_quality(data):
    """简单的偏好数据质量检查"""
    issues = []
    stats = {"total": len(data), "good": 0, "warnings": 0}
    
    for i, item in enumerate(data):
        # 检查长度差异是否过大（可能是长度偏差）
        len_ratio = len(item["chosen"]) / max(len(item["rejected"]), 1)
        if len_ratio > 5:
            issues.append(f"  样本{i}: chosen/rejected 长度比={len_ratio:.1f}x (可能有长度偏差)")
            stats["warnings"] += 1
        
        # 检查是否有空字段
        if not item["prompt"].strip() or not item["chosen"].strip() or not item["rejected"].strip():
            issues.append(f"  样本{i}: 存在空字段")
            stats["warnings"] += 1
        
        # 检查 chosen 和 rejected 是否相同
        if item["chosen"].strip() == item["rejected"].strip():
            issues.append(f"  样本{i}: chosen 和 rejected 完全相同")
            stats["warnings"] += 1
        else:
            stats["good"] += 1
    
    print(f"数据质量报告: {stats['good']}/{stats['total']} 样本合格")
    if issues:
        print("警告:")
        for issue in issues:
            print(issue)
    return stats

check_preference_quality(preference_examples)

---
## 3. DPO 实战训练

In [ ]:
# 检查依赖
import importlib

required = ["transformers", "trl", "peft", "datasets"]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  {pkg}: OK")
    except ImportError:
        missing.append(pkg)
        print(f"  {pkg}: 缺失")

if missing:
    print(f"\n请安装: pip install {' '.join(missing)}")
else:
    print("\n所有依赖已就绪!")

In [ ]:
# 3.1 加载 SFT 模型
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2-0.5B"  # 使用较小模型方便演示
print(f"加载模型: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    trust_remote_code=True,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,}")
print(f"设备: {device}")

In [ ]:
# 3.2 准备偏好数据
from datasets import Dataset

# 合成偏好数据（实际项目中应使用真实数据）
templates = [
    {
        "prompt": "请解释什么是机器学习？",
        "chosen": "机器学习是人工智能的一个分支，它使计算机系统能够从数据中自动学习和改进，而无需显式编程。主要分为监督学习、无监督学习和强化学习三大类。",
        "rejected": "机器学习就是让机器学东西。",
    },
    {
        "prompt": "Python 和 Java 有什么区别？",
        "chosen": "Python 和 Java 的主要区别包括：1) 类型系统：Python 是动态类型，Java 是静态类型；2) 语法：Python 语法简洁，使用缩进，Java 使用大括号；3) 执行方式：Python 解释执行，Java 编译为字节码在 JVM 上运行。",
        "rejected": "一个用缩进一个用大括号。",
    },
    {
        "prompt": "如何提高编程能力？",
        "chosen": "提高编程能力的有效方法：1) 坚持每日编程练习；2) 阅读优秀开源项目的源代码；3) 参与开源项目贡献；4) 系统学习数据结构与算法；5) 构建个人项目解决实际问题。",
        "rejected": "多写代码就行了。",
    },
    {
        "prompt": "什么是深度学习？",
        "chosen": "深度学习是机器学习的一个子领域，使用多层神经网络来学习数据的层次化表示。核心组件包括卷积层、循环层、注意力机制等。在计算机视觉和自然语言处理等领域取得了突破性成果。",
        "rejected": "就是很多层的神经网络。",
    },
    {
        "prompt": "请给我一个排序算法的例子。",
        "chosen": "快速排序是一种高效的排序算法，平均时间复杂度为 O(n log n)。它使用分治策略，选择一个基准元素将数组分为两部分，递归排序。Python实现：def quicksort(arr): ...",
        "rejected": "用 sort() 就行。",
    },
]

# 扩充数据
num_samples = 200
data = {"prompt": [], "chosen": [], "rejected": []}
for i in range(num_samples):
    t = templates[i % len(templates)]
    suffix = f"（变体 {i+1}）" if i >= len(templates) else ""
    data["prompt"].append(t["prompt"] + suffix)
    data["chosen"].append(t["chosen"])
    data["rejected"].append(t["rejected"])

train_dataset = Dataset.from_dict(data)
print(f"偏好数据集大小: {len(train_dataset)}")
print(f"样本字段: {train_dataset.column_names}")
print(f"\n示例:")
print(f"  prompt:   {train_dataset[0]['prompt']}")
print(f"  chosen:   {train_dataset[0]['chosen'][:60]}...")
print(f"  rejected: {train_dataset[0]['rejected']}")

In [ ]:
# 3.3 配置 LoRA
from peft import LoraConfig, TaskType

peft_config = LoraConfig(
    r=8,                # 低秩矩阵的秩
    lora_alpha=16,      # 缩放因子，通常设为 r 的 2 倍
    lora_dropout=0.05,  # dropout 防止过拟合
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # 对注意力层应用 LoRA
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

print(f"LoRA 配置:")
print(f"  rank (r):      {peft_config.r}")
print(f"  alpha:         {peft_config.lora_alpha}")
print(f"  target:        {peft_config.target_modules}")

In [ ]:
# 3.4 配置 DPOTrainer
from trl import DPOTrainer, DPOConfig

output_dir = "./dpo_output"

dpo_config = DPOConfig(
    output_dir=output_dir,
    num_train_epochs=1,                 # 训练轮数
    per_device_train_batch_size=2,       # batch size
    gradient_accumulation_steps=4,       # 梯度累积
    learning_rate=5e-5,                  # 学习率
    beta=0.1,                            # DPO 温度参数 β
    logging_steps=5,                     # 每 5 步记录
    save_steps=50,                       # 每 50 步保存
    max_length=512,                      # 最大总长度
    max_prompt_length=256,               # prompt 最大长度
    remove_unused_columns=False,
    fp16=(device == "cuda"),
    report_to="none",
)

print(f"DPO 超参数:")
print(f"  beta (β):          {dpo_config.beta}  ← 控制保守程度")
print(f"  learning_rate:     {dpo_config.learning_rate}")
print(f"  batch_size:        {dpo_config.per_device_train_batch_size}")
print(f"  grad_accum_steps:  {dpo_config.gradient_accumulation_steps}")
print(f"  effective_batch:   {dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps}")

In [ ]:
# 3.5 SFT 基线评估（训练前）
test_prompts = [
    "请解释什么是强化学习？",
    "如何学好数学？",
    "Python 的装饰器是什么？",
]

print("=" * 60)
print("SFT 基线评估（DPO 训练前）")
print("=" * 60)

sft_responses = []
model.eval()
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=100, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    sft_responses.append(response)
    print(f"\nPrompt: {prompt}")
    print(f"SFT 回复: {response[:200]}")

In [ ]:
# 3.6 初始化并运行 DPOTrainer
print("初始化 DPOTrainer...")

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,  # DPOTrainer 会自动应用 LoRA
)

# 打印可训练参数
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LoRA 可训练参数: {trainable:,} ({100*trainable/total_params:.2f}%)")

print("\n开始 DPO 训练...")
train_result = dpo_trainer.train()

# 记录训练指标
print("\n训练指标:")
for k, v in train_result.metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# 3.7 DPO 训练后评估
print("=" * 60)
print("SFT + DPO 模型评估（训练后）")
print("=" * 60)

dpo_responses = []
model.eval()
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=100, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    dpo_responses.append(response)
    print(f"\nPrompt: {prompt}")
    print(f"DPO 回复: {response[:200]}")

In [ ]:
# 3.8 Reward Margin 可视化
# 模拟训练过程中的 reward margin 变化

np.random.seed(42)
num_steps = 100
beta = 0.1

# 模拟 chosen 和 rejected 的对数概率比
log_ratio_chosen = np.zeros(num_steps)
log_ratio_rejected = np.zeros(num_steps)
losses = np.zeros(num_steps)
margins = np.zeros(num_steps)

for step in range(num_steps):
    log_ratio_chosen[step] = 0.5 * np.log(step + 1) + np.random.randn() * 0.1
    log_ratio_rejected[step] = -0.3 * np.log(step + 1) + np.random.randn() * 0.1
    diff = beta * (log_ratio_chosen[step] - log_ratio_rejected[step])
    losses[step] = -np.log(1 / (1 + np.exp(-diff)) + 1e-10)
    margins[step] = beta * (log_ratio_chosen[step] - log_ratio_rejected[step])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 损失曲线
axes[0].plot(losses, color="blue", alpha=0.7)
axes[0].set_title("DPO 损失曲线", fontweight="bold")
axes[0].set_xlabel("训练步数")
axes[0].set_ylabel("损失")
axes[0].grid(alpha=0.3)

# Chosen vs Rejected
axes[1].plot(log_ratio_chosen, color="green", label="Chosen log-ratio (应该上升)", alpha=0.7)
axes[1].plot(log_ratio_rejected, color="red", label="Rejected log-ratio (应该下降)", alpha=0.7)
axes[1].axhline(y=0, color="k", ls="--", alpha=0.3)
axes[1].set_title("Chosen vs Rejected 对数概率比", fontweight="bold")
axes[1].set_xlabel("训练步数")
axes[1].set_ylabel("log π_θ / π_ref")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# Reward Margin
axes[2].plot(margins, color="purple", alpha=0.7)
axes[2].axhline(y=0, color="k", ls="--", alpha=0.3)
axes[2].fill_between(range(num_steps), margins, 0, alpha=0.2,
                     where=np.array(margins) > 0, color="green", label="正 margin (好)")
axes[2].fill_between(range(num_steps), margins, 0, alpha=0.2,
                     where=np.array(margins) <= 0, color="red", label="负 margin (差)")
axes[2].set_title("Reward Margin (越大越好)", fontweight="bold")
axes[2].set_xlabel("训练步数")
axes[2].set_ylabel("β * (r_chosen - r_rejected)")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.suptitle("DPO 训练过程监控", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. 效果对比：Base vs SFT vs DPO

In [ ]:
# 三方对比表格
print("=" * 80)
print("Base vs SFT vs DPO 对比")
print("=" * 80)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'─' * 70}")
    print(f"Prompt: {prompt}")
    print(f"{'─' * 70}")
    print(f"  Base:  (预训练模型通常不会直接回答问题，可能续写或重复)")
    print(f"  SFT:   {sft_responses[i][:100]}...")
    print(f"  DPO:   {dpo_responses[i][:100]}...")

### 三种模型的定性对比

| 维度 | Base 模型 | SFT 模型 | SFT + DPO 模型 |
|------|----------|----------|----------------|
| **指令遵循** | 差（可能续写而非回答） | 好（学会了对话格式） | 好 |
| **回复质量** | 不可控 | 中等（模仿标注数据） | 较高（偏好优化） |
| **安全性** | 低 | 中 | 较高 |
| **谄媚程度** | N/A | 较高 | 较低 |
| **拒绝能力** | 无 | 有限 | 较强 |

In [ ]:
# 可视化对比
fig, ax = plt.subplots(figsize=(10, 6))

categories = ["指令遵循", "回复质量", "安全性", "抗谄媚", "知识准确"]
base_scores =  [1, 2, 1, 2, 3]
sft_scores =   [4, 3, 3, 2, 3]
dpo_scores =   [4, 4, 4, 4, 3]

x = np.arange(len(categories))
width = 0.25

ax.bar(x - width, base_scores, width, label="Base", color="#95A5A6", alpha=0.8)
ax.bar(x, sft_scores, width, label="SFT", color="#FFB347", alpha=0.8)
ax.bar(x + width, dpo_scores, width, label="SFT + DPO", color="#4ECDC4", alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("评分 (1-5)")
ax.set_title("Base vs SFT vs DPO 模型能力对比", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 5.5)

plt.tight_layout()
plt.show()

In [ ]:
# β 对 DPO 损失的影响
fig, ax = plt.subplots(figsize=(10, 6))

x = np.linspace(-5, 5, 300)  # 隐式奖励差
betas = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]

for b in betas:
    y = -np.log(1 / (1 + np.exp(-b * x)) + 1e-10)
    ax.plot(x, y, label=f"β={b}", lw=1.5)

ax.set_xlabel("隐式奖励差 (r_chosen - r_rejected)", fontsize=11)
ax.set_ylabel("DPO 损失", fontsize=11)
ax.set_title("β 对 DPO 损失形状的影响", fontsize=14, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.axvline(x=0, color="gray", ls=":", alpha=0.5)

ax.text(3, 2, "β 大 → 损失更陡\n→ 更保守", fontsize=10, color="gray")
ax.text(-4, 0.5, "β 小 → 损失更平\n→ 更激进", fontsize=10, color="gray")

plt.tight_layout()
plt.show()

---
## 5. 总结

### DPO vs RLHF 对比表

| 维度 | RLHF | DPO |
|------|------|-----|
| **训练阶段** | 3 阶段（SFT → RM → PPO） | 2 阶段（SFT → DPO） |
| **需要 RM** | 是 | 否（隐式奖励） |
| **需要 PPO** | 是 | 否 |
| **稳定性** | 低（PPO 训练不稳定） | 高（标准监督学习） |
| **内存需求** | 高（4个模型） | 中（2个模型：策略+参考） |
| **超参数** | 多（PPO + RM + KL） | 少（主要是 β） |
| **数据需求** | 偏好对 + prompt | 偏好对 |
| **理论保证** | 近似最优 | 等价最优（在理想条件下） |
| **实际效果** | 大规模验证（GPT-4, Claude） | 小模型上接近 RLHF |
| **工业采用** | 头部公司 | 广泛（Llama, Zephyr 等） |

### 思考题

1. **DPO 需要参考模型（内存翻倍），有没有办法去掉参考模型？** 提示：思考 ORPO 和 SimPO。

2. **DPO 对数据质量非常敏感，如果偏好数据有很多噪声会怎样？** 你会如何改进？

3. **β 的选择对 DPO 至关重要。** 如果 β 设太小（如 0.01），模型行为会有什么变化？太大（如 1.0）呢？

---
*下一课：Day 08 — 对齐技术全景图，从 RLHF/DPO 到 KTO/ORPO/SimPO/GRPO*